In [8]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
import pickle

# -----------------------------
# NLTK SETUP
# -----------------------------
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

# -----------------------------
# TEXT CLEANING
# -----------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)       # remove URLs
    text = re.sub(r"@\w+", "", text)                 # remove mentions
    text = re.sub(r"[^a-zA-Z\s]", "", text)          # keep letters only
    text = re.sub(r"\s+", " ", text).strip()         # remove extra spaces
    # remove stopwords + stemming
    text = " ".join([ps.stem(word) for word in text.split() if word not in stop_words])
    return text

# -----------------------------
# LOAD DATASET
# -----------------------------
df = pd.read_csv("train.csv")  # your dataset
df["clean_tweet"] = df["tweet"].apply(clean_text)

# -----------------------------
# ADD EXTRA NEUTRAL/ POSITIVE EXAMPLES
# -----------------------------
neutral_examples = [
    ("I love you", 2),
    ("You are amazing", 2),
    ("Have a great day", 2),
    ("I appreciate you", 2),
    ("Good morning everyone", 2),
    ("Happy birthday!", 2),
    ("Congratulations on your success", 2),
    ("Thank you for your help", 2),
    ("You did a great job", 2),
    ("Peace and love to all", 2)
]

df_extra = pd.DataFrame(neutral_examples, columns=["tweet", "class"])
df_extra["clean_tweet"] = df_extra["tweet"].apply(clean_text)
df = pd.concat([df, df_extra], ignore_index=True)

# -----------------------------
# FEATURES AND TARGET
# -----------------------------
X = df["clean_tweet"]
y = df["class"]

# -----------------------------
# TF-IDF VECTORIZATION
# -----------------------------
vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(X)

# -----------------------------
# BALANCE DATA WITH OVERSAMPLING
# -----------------------------
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X_vec, y)

# -----------------------------
# TRAIN-TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42
)

# -----------------------------
# TRAIN LOGISTIC REGRESSION MODEL
# -----------------------------
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# -----------------------------
# EVALUATION
# -----------------------------
pred = model.predict(X_test)
print("📌 Accuracy:", accuracy_score(y_test, pred))
print("\n📌 Classification Report:\n", classification_report(y_test, pred))
print("\n📌 Confusion Matrix:\n", confusion_matrix(y_test, pred))

# -----------------------------
# SAVE MODEL AND VECTORIZER
# -----------------------------
pickle.dump(model, open("hate_speech_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

print("\n🎉 Model and vectorizer saved successfully!")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


📌 Accuracy: 0.9333854438075386

📌 Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.96      0.93      3848
           1       0.95      0.85      0.90      3794
           2       0.95      0.98      0.96      3872

    accuracy                           0.93     11514
   macro avg       0.93      0.93      0.93     11514
weighted avg       0.93      0.93      0.93     11514


📌 Confusion Matrix:
 [[3702  135   11]
 [ 351 3243  200]
 [  24   46 3802]]

🎉 Model and vectorizer saved successfully!
